# Monte Carlo of Lennard–Jones Fluids 

## Theory: 
This project primarily focuses on using Monte Carlo (MC) simulations to explore gas-liquid transitions and phase equilibria, particularly for systems involving Lennard-Jones (LJ) fluids.
MC simulations are a powerful computational technique widely used for random sampling in complex systems. A key algorithm used in these simulations is the Metropolis criterion, which determines whether to accept or reject proposed moves based on changes in the system’s energy and other thermodynamic properties, such as pressure or temperature. By using this equation, the system is able to experiment with several configurations until reaching equilibrium.
In these simulations, particles in a box are moved randomly, and their interactions are evaluated using the LJ potential, which captures both attractive and repulsive forces between molecules. The balance between these two forces leads to a potential energy curve where particles attract each other at longer distances but strongly repel each other when they are very close.
The simulation provides useful insights into phase equilibria by replicating the behavior of fluids across diverse conditions, such as varying temperature and pressure. It offers a simplified but effective approach to studying molecular interactions and phase behavior in systems where exact analytical solutions are challenging to obtain.

- **Lennard–Jones reduced units:** set σ = 1 for length and ε = 1 for energy. Temperature is 
$$T^* = k_B T / \varepsilon,$$
pressure is $$p^* = p\,\sigma^3/\varepsilon,$$ density is $$\rho^* = N\,\sigma^3/V.$$
- **Potential energy:** reduced form $$U^* = 4\left[(\sigma/r)^{12} - (\sigma/r)^6\right]$$ scaled by ε.
- NVT uses exp(-ΔU) (implicit β=1); NPT uses Ua/Ur with 1/T* scaling; cutoff via r² ≥ 2.5.

## Imports & Parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Parameters
N = 1000  # Number of particles
rho = 0.74  # Density liq  = 0.74, Density gas = 0.023
L = (N / rho)**(1/3)  # Length of the cubic box
epsilon = 1.05  # Energy scale based on the condition
sigma = 1.0  # Distance scale
kB = 1.0  # Boltzmann constant
T = 1  # Temperature
num_iterations = 500000  # Can be adjusted with our need (iters_nvt & iters_npt)
delta = 0.5  # Maximum displacement for Monte Carlo move Liquid
iters_nvt=500000
seed=0
p=0.023
T_star=1/1.05
iters_npt=1000000
bins=50


### Periodic Boundary Conditions (PBC)

In [ ]:
# Periodic boundary conditions (PBC). If a particle crosses the boundary, it re-enters from the opposite side.
def apply_pbc(dx, dy, dz, L):
    dx = dx - np.round(dx / L) * L
    dy = dy - np.round(dy / L) * L
    dz = dz - np.round(dz / L) * L
    return dx, dy, dz


### Lennard–Jones Energy

In [ ]:
# Lennard-Jones energy using your r^2 cutoff behavior
def U_full_slicing(x, y, z, L):
    result = 0.0
    for d in range(len(x) - 1):
        dx = x[d] - x[d+1:]
        dy = y[d] - y[d+1:]
        dz = z[d] - z[d+1:]
        # Apply periodic boundary conditions (PBC)
        dx, dy, dz = apply_pbc(dx, dy, dz, L)
        # Lennard-Jones potential is used to calculate the interaction energy between particles
        r2 = dx**2 + dy**2 + dz**2
        phi = 4 * epsilon * ((sigma**12 / r2**6) - (sigma**6 / r2**3))
        phi[r2 >= 2.5] = 0  # Safety threshold to ignore large distances
        result += np.sum(phi)
    return result

# Calculating energy for one particle (used in Metropolis dU)
def U_for_one_particle(x, y, z, L, identity):
    # Displacements between particle `identity` and all others
    mask = np.arange(len(x)) != identity
    dx = x[identity] - x[mask]
    dy = y[identity] - y[mask]
    dz = z[identity] - z[mask]
    # Apply PBC
    dx, dy, dz = apply_pbc(dx, dy, dz, L)
    # Distances
    r2 = dx**2 + dy**2 + dz**2
    phi = np.zeros_like(r2)
    # Mask for valid distances under cutoff
    new_mask = r2 < 2.5
    # Calculate Lennard-Jones potential (ε fixed at 1.05)
    phi[new_mask] = 4 * 1.05 * ((sigma**12 / r2[new_mask]**6) - (sigma**6 / r2[new_mask]**3))
    return np.sum(phi[new_mask])


### NVT Monte Carlo (liquid)

In [ ]:
# NVT Metropolis Monte Carlo (liquid)
np.random.seed(seed)  # For reproducibility

# Initialize particle positions randomly
x = (np.random.rand(N)-0.5) * L  # array of N random numbers from -1L to 1L
y = (np.random.rand(N)-0.5) * L
z = (np.random.rand(N)-0.5) * L

# Initial energy
U_initial = U_full_slicing(x, y, z, L)
U_last = U_initial

# Lists to store energy values for plotting
energy_values = []
energy_list = []
energy_array = np.array(energy_list)
energy_interval = 1000  # How often to store energy (just to check)

# Initialize to store coordinates during sampling
sampled_coordinates_liq = []

# Number of sampling runs after equilibrium
num_samples = 10  # Sample 10 snapshots after ~equilibrium

for iteration in range(num_iterations):
    # Choose a random particle
    identity = np.random.randint(N)
    U_oldp = U_for_one_particle(x, y, z, L, identity)

    # Save the current position
    old_x, old_y, old_z = x[identity], y[identity], z[identity]

    # Propose a new position
    x[identity] += (np.random.rand() - 0.5) * delta
    y[identity] += (np.random.rand() - 0.5) * delta
    z[identity] += (np.random.rand() - 0.5) * delta

    # Apply PBC
    x[identity], y[identity], z[identity] = apply_pbc(x[identity], y[identity], z[identity], L)

    # Energy change for the selected particle
    U_newp = U_for_one_particle(x, y, z, L, identity)
    delta_U = U_newp - U_oldp

    # Metropolis acceptance criterion
    if np.random.rand() < np.exp(-delta_U):
        # Accept the move
        U_last += delta_U
    else:
        # Reject the move and revert to the old position
        x[identity], y[identity], z[identity] = old_x, old_y, old_z

    # Update full energy occasionally (10% of the time)
    if iteration % (num_iterations//10) == 0:
        U_last = U_full_slicing(x, y, z, L)

    # Append current U to energy array
    energy_array = np.append(energy_array, U_last)

    # Store energy every energy_interval iterations
    if iteration % energy_interval == 0:
        energy_values.append(U_last)

    # Store coordinates during the final 10% of runs (sampling snapshots)
    if iteration >= (9*num_iterations//10) and len(sampled_coordinates_liq) < num_samples:
        sampled_coordinates_liq.append(np.vstack((x, y, z)).T)  # store x, y, z for all particles

print(f"Final energy: {U_last:.2f}")


### Energy Plot (NVT)

In [ ]:
# Plotting the energy values
plt.figure(figsize=(10, 5))
plt.plot(np.arange(0, num_iterations, energy_interval), energy_values)
plt.xlabel('Iteration')
plt.ylabel('Energy')
plt.title('Energy vs Iteration for Monte Carlo Simulation (NVT, liquid)')
plt.grid()
plt.axhline(y=np.min(energy_values), linestyle='--', label='Minimum Energy')
plt.legend()
plt.show()

# Average energy per particle over tail
mean_U_pp = round(np.mean(energy_array[-max(1, num_iterations//5):])/N, 3)
print(f"{mean_U_pp} average units of energy per particle in liquid phase")


### Radial Distribution Function g(r)

In [ ]:
# Radial distribution function g(r) (NVT, liquid)
# PBC for g(r) uses the same apply_pbc defined above.
def g_r(coordinates, L, dr, rho=None, rcutoff=0.9):
    Nloc = len(coordinates)
    # Max radius to half the box, scaled by rcutoff
    r_max = (L / 2) * rcutoff
    radii = np.arange(dr, r_max, dr)  # radii values for each bin
    n_radii = len(radii)

    # If density not provided, compute from box
    if rho is None:
        rho = Nloc / (L ** 3)

    gvals = np.zeros(n_radii)  # RDF array

    # Loop over pairs
    for i in range(Nloc):
        for j in range(i + 1, Nloc):
            dx = coordinates[i, 0] - coordinates[j, 0]
            dy = coordinates[i, 1] - coordinates[j, 1]
            dz = coordinates[i, 2] - coordinates[j, 2]
            dx, dy, dz = apply_pbc(dx, dy, dz, L)
            r = np.sqrt(dx**2 + dy**2 + dz**2)
            if r < r_max:
                bin_idx = int(r / dr)
                if bin_idx < n_radii:
                    gvals[bin_idx] += 2  # count i-j and j-i

    # Normalize by ideal gas in shell volume
    for k, r in enumerate(radii):
        shell_vol = (4/3) * np.pi * ((r + dr)**3 - r**3)
        ideal = rho * shell_vol * Nloc
        if ideal > 0:
            gvals[k] /= ideal

    return gvals, radii

# Average g(r) across sampled snapshots
if len(sampled_coordinates_liq) > 0:
    dr = 0.05
    rho_liq = rho
    n_samples = len(sampled_coordinates_liq)
    # find minimum length for safe averaging
    min_len = None
    first_radii = None
    temp_list = []
    for coords in sampled_coordinates_liq:
        g_sample, r_sample = g_r(coords, L, dr, rho=rho_liq)
        temp_list.append((g_sample, r_sample))
        min_len = len(g_sample) if min_len is None else min(min_len, len(g_sample))
        if first_radii is None: first_radii = r_sample
    g_sum = np.zeros(min_len)
    for g_s, r_s in temp_list:
        g_sum += g_s[:min_len]
    g_avg = g_sum / n_samples
    radii_avg = first_radii[:min_len]

    plt.figure(figsize=(8,5))
    plt.plot(radii_avg, g_avg, label='Averaged g(r) (liquid)')
    plt.xlabel('r / sigma'); plt.ylabel('g(r)'); plt.title('Radial Distribution Function (NVT, liquid)')
    plt.grid(); plt.legend(); plt.show()
else:
    print('No sampled snapshots collected; run NVT first.')


### NPT: P(V) & Density

In [ ]:
# NPT volume-move simulation (liquid) — Ua/Ur split and acceptance as in your project
# Use final coordinates from NVT run
x_liq = x.copy(); y_liq = y.copy(); z_liq = z.copy()
V_init = N / rho
dV_max = V_init * 0.03  # Liquid = 3%

# Attractive/repulsive split
def U_system_divided(xv, yv, zv, U_att, U_rep):
    for d in range(len(xv) - 1):  # Calc. attr. and repul. energies
        dx = xv[d] - xv[d+1:]
        dy = yv[d] - yv[d+1:]
        dz = zv[d] - zv[d+1:]
        dx, dy, dz = apply_pbc(dx, dy, dz, L)
        r = np.sqrt(dx**2 + dy**2 + dz**2)
        Ua = -4 * 1.05 * (1/T_star) * (sigma / r)**6
        Ur =  4 * 1.05 * (1/T_star) * (sigma / r)**12
        U_att += np.sum(Ua)
        U_rep += np.sum(Ur)
    return U_att, U_rep

# Initialize
V_old = V_init
U_attract = 0.0; U_repuls = 0.0
U_attract, U_repuls = U_system_divided(x_liq, y_liq, z_liq, U_attract, U_repuls)

V_array = np.array([V_old])
P_V = np.zeros(bins)  # histogram
minV = 0.8*V_init; maxV = 1.2*V_init  # a safe range for bin labeling

for iteration in range(iters_npt):
    # propose volume move
    dV = (np.random.rand() - 0.5) * dV_max
    V_prop = V_old + dV
    if V_prop <= 0:  # reject non-physical
        V_array = np.append(V_array, V_old)
        continue
    L_prop = V_prop**(1/3)
    Ua_prop = U_attract * (L / L_prop)**6
    Ur_prop = U_repuls * (L / L_prop)**12
    dUa = Ua_prop - U_attract
    dUr = Ur_prop - U_repuls
    # acceptance
    acc = np.exp(-(dUa + dUr + p*dV) + N * np.log(V_prop / V_old))
    if np.random.rand() < acc:
        # accept
        x_liq *= (L_prop / L); y_liq *= (L_prop / L); z_liq *= (L_prop / L)
        V_old = V_prop; L = L_prop
        U_attract = Ua_prop; U_repuls = Ur_prop

    # sample volume
    V_array = np.append(V_array, V_old)
    # histogram bin
    b = int(np.floor(((V_old - minV) / (maxV - minV)) * bins)); b = max(0, min(b, bins - 1))
    P_V[b] += 1

    # resample Ua/Ur occasionally
    if iteration % max(1, iters_npt//10) == 0:
        U_attract, U_repuls = U_system_divided(x_liq, y_liq, z_liq, U_attract, U_repuls)

# normalize histogram to probability
if P_V.sum() > 0:
    P_V = P_V / P_V.sum()


In [ ]:
# Build volume-axis labels and plot P(V)
bin_lab = np.linspace(0.8*(N/rho), 1.2*(N/rho), bins, endpoint=False)
plt.figure(figsize=(8,5))
plt.plot(bin_lab, P_V)
plt.xlabel('Volume (bin)')
plt.ylabel('Probability')
plt.title('P(V) histogram (NPT, liquid)')
plt.grid(); plt.show()

# Density distribution
density_array_liq = N / V_array
plt.figure(figsize=(8,5))
plt.hist(density_array_liq, bins=30, density=True, alpha=0.7, label='Density Distribution')
plt.xlabel('Density'); plt.ylabel('Probability Density'); plt.title('Density Distribution (NPT, liquid)')
plt.legend(); plt.grid(); plt.show()
